# 第 1 章 §1.3 · Prompt Chaining

书里这个脚本做两件事：先从一段自然语言里**抽出规格**，再把规格**转成 JSON**——
两次独立调用，中间用一个 dict 的键名接起来。

kernel 选 `Python (agentic-book)`。代码与书一致，只有 **【适配】** 处不同。
六个关键点、执行时序表、生产化缺口表都在 `../../my_notes/ch01_prompt_chaining.md`，
想动手拆就用 `01_3_debug_lab.ipynb`。

In [1]:
# 【适配】model：书里 ChatOpenAI 默认 gpt-3.5-turbo，DeepSeek 不认
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 注意：这一行不发任何网络请求，只是构造客户端对象。
llm = ChatOpenAI(temperature=0, model="deepseek-flash")

# ★ 本章新增 ChatPromptTemplate：纯本地字符串插值，把变量填进模板，不花 token。
#   {text_input} 被登记为输入变量，少传抛 KeyError，多传被静默忽略。
prompt_extract = ChatPromptTemplate.from_template(
    "Extract the technical specifications from the following text:\n\n{text_input}"
)

# 它的变量叫 {specifications}，必须和下面 dict 的键名完全一致 —— 这就是两段之间的接口契约。
prompt_transform = ChatPromptTemplate.from_template(
    "Transform the following specifications into a JSON object with 'cpu', 'memory', and 'storage' as keys:\n\n{specifications}"
)

# ★ 本章新增 `|`：Python 按位或被重载成「串联」，等价于 RunnableSequence([...])。
#   这一行不执行任何东西，只是构造 Runnable 对象。
extraction_chain = prompt_extract | llm | StrOutputParser()

# ★ 本章新增 单键 dict：LangChain 自动转成 RunnableParallel，语义是
#   把上游输入原封不动喂给 extraction_chain，再把返回值装进键 "specifications"。
#   键名把两段粘在一起，而这个契约完全不受校验保护。
full_chain = (
    {"specifications": extraction_chain}
    | prompt_transform
    | llm
    | StrOutputParser()
)

input_text = "The new laptop model features a 3.5 GHz octa-core processor, 16GB of RAM, and a 1TB NVMe SSD."

# invoke() 才是点火。内部：第 1 次 API 调用（抽取）→ dict 打包 → 第 2 次 API 调用（转 JSON）。
# 两次严格串行 —— 这就是 chaining 的本质，也是它的代价（延迟叠加、错误累积）。
final_result = full_chain.invoke({"text_input": input_text})

# final_result 是【字符串】，不是 dict。模型吐什么就打印什么。
print("\n--- Final JSON Output ---")
print(final_result)


--- Final JSON Output ---
{
  "cpu": "3.5 GHz octa-core",
  "memory": "16GB",
  "storage": "1TB NVMe SSD"
}


In [2]:
# 证明「看起来像 JSON」≠「是 JSON」。这格不调用模型。
import json

print("type(final_result) =", type(final_result).__name__)
print("是 str 吗          =", isinstance(final_result, str))
print("是 dict 吗         =", isinstance(final_result, dict))

try:
    parsed = json.loads(final_result)
    print("\njson.loads 成功 ->", type(parsed).__name__, "=", parsed)
except json.JSONDecodeError as e:
    print("\njson.loads 失败 ->", e)
    print("原始输出：")
    print(final_result)
    print("\n模型这次把 JSON 包在了 Markdown ``` 围栏里，得先剥掉才能解析 —— 这就是没有校验的代价。")

type(final_result) = TextAccessor
是 str 吗          = True
是 dict 吗         = False

json.loads 成功 -> dict = {'cpu': '3.5 GHz octa-core', 'memory': '16GB', 'storage': '1TB NVMe SSD'}
